<a href="https://colab.research.google.com/github/mar-olya/compling-Markovich/blob/main/%D0%9C%D0%B0%D1%80%D0%BA%D0%BE%D0%B2%D0%B8%D1%87_w2v_hw_ipynb%22.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

В этом практикуме мы рассмотрим работу с библиотекой **Gensim** для работы с векторными представлениями текста

Мы рассмотрим
- **Word2Vec** - векторные представления слов
- **FastText** - улучшенные представления с учетом морфологии  
- **Doc2Vec** - векторные представления документов


In [1]:
!pip install gensim

import gensim
import gensim.downloader as api
from gensim.models import Word2Vec, FastText, Doc2Vec
from gensim.models.doc2vec import TaggedDocument
import numpy as np

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 43.9 MB/s eta 0:00:00


## Часть 1: Word2Vec

### Что такое Word2Vec?

Word2Vec преобразует слова в векторы чисел так, что семантически похожие слова оказываются близко в векторном пространстве.

**Два основных алгоритма:**
- **CBOW** - предсказывает слово по контексту
- **Skip-gram** - предсказывает контекст по слову

**Загрузка предобученной модели**

In [2]:
w2v_model = api.load('glove-wiki-gigaword-100')

print(f"Размер словаря: {len(w2v_model.key_to_index)}")
print(f"Размерность векторов: {w2v_model.vector_size}")

[==================================================] 100.0% 128.1/128.1MB downloaded
Размер словаря: 400000
Размерность векторов: 100


Найдите документацию `gensim`: какие датасеты кроме `glove-wiki-gigaword-100` доступны в библиотеке?

Выберите 3 датасета и кратко опишите их (источник данных, примерный объем, зачем такой датасет может использоваться)

`glove-twitter-200`	- Твиттер (2 млрд. твитов, 27 млрд токенов, 1.2 миллиона токенов без учета регистра)
- Классификация твитов (спам/не спам, позитив/негатив)
- Анализ настроений (sentiment analysis)
- Обнаружение ботов и фейковых аккаунтов
- Тренд-анализ и виральный контент

`word2vec-ruscorpora-300`	- НКРЯ (около 250 млн. слов)
- Машинный перевод RU↔EN
- Русская классификация текстов
- Анализ русских соцсетей (ВК, Telegram)
- Чат-боты на русском языке

`word2vec-google-news-300`	- Google News (около 100 млрд. слов)
- Классификация новостей (политика, спорт, технологии)
- Извлечение именованных сущностей (NER)
- Анализ медиа-контента
- Поисковые системы

**Базовые операции с векторами**

In [3]:
# Получаем вектор слова
vector = w2v_model['computer']
print(f"Вектор слова 'computer': {vector[:5]}...")  # Показываем первые 5 чисел

# Вычисляем схожесть между словами
similarity = w2v_model.similarity('computer', 'laptop')
print(f"Схожесть 'computer' и 'laptop': {similarity:.4f}")

Вектор слова 'computer': [-0.16298   0.30141   0.57978   0.066548  0.45835 ]...
Схожесть 'computer' и 'laptop': 0.7024


**Поиск похожих слов**

In [4]:
# Находим похожие слова
similar_words = w2v_model.most_similar('python', topn=5)
print("Слова, похожие на 'python':")
for word, score in similar_words:
    print(f"  {word}: {score:.4f}")

Слова, похожие на 'python':
  monty: 0.6886
  php: 0.5865
  perl: 0.5784
  cleese: 0.5447
  flipper: 0.5113


*Ваш ответ здесь*

**Задание**

1. Загрузите любой датасет из gensim на ваш выбор

In [5]:
w2v_model = api.load('word2vec-ruscorpora-300')

print(f"Размер словаря: {len(w2v_model.key_to_index)}")
print(f"Размерность векторов: {w2v_model.vector_size}")

[==================================================] 100.0% 198.8/198.8MB downloaded
Размер словаря: 184973
Размерность векторов: 300


2. Напишите функцию, которая принимает на вход любое слово и вовращает 10 наиболее близких по вектору слов

In [12]:
# Находим похожие слова
similar_words = w2v_model.most_similar('мама_NOUN', topn=10)
print("Слова, похожие на 'мама_NOUN':")
for word, score in similar_words:
    print(f"  {word}: {score:.4f}")

Слова, похожие на 'мама_NOUN':
  бабушка_NOUN: 0.7865
  мамочка_NOUN: 0.7664
  мамин_ADJ: 0.7359
  катя_NOUN: 0.7299
  тетя_NOUN: 0.7164
  пидет_ADV: 0.7119
  папа_NOUN: 0.7103
  тетя::минзамал_NOUN: 0.6881
  оля_NOUN: 0.6815
  марь::юрьн_NOUN: 0.6714


3. Обучите модель Word2Vec на тестовом датасете из ячейки ниже

Примените следующие настройки:

- размер вектора: 50
- размер окна: 3
- минимальная частота слова: 1
- потоков: 2
- использовать skip-gram

In [13]:
cooking_sentences = [
    ['варить', 'суп', 'овощи', 'морковь', 'картофель'],
    ['жарить', 'курица', 'сковорода', 'масло', 'специи'],
    ['печь', 'хлеб', 'мука', 'дрожжи', 'духовка'],
    ['резать', 'овощи', 'салат', 'помидоры', 'огурцы'],
    ['смешивать', 'ингредиенты', 'тесто', 'яйца', 'молоко'],
    ['варить', 'паста', 'вода', 'соль', 'соус'],
    ['гриль', 'мясо', 'овощи', 'уголь', 'барбекю'],
    ['тушить', 'говядина', 'горшок', 'вино', 'травы'],
    ['запекать', 'рыба', 'лимон', 'духовка', 'фольга'],
    ['готовить', 'завтрак', 'яичница', 'бекон', 'тост'],
    ['месить', 'тесто', 'пирог', 'начинка', 'яблоки'],
    ['кипятить', 'вода', 'чай', 'кофе', 'чашка'],
    ['мариновать', 'мясо', 'соус', 'специи', 'холодильник'],
    ['взбивать', 'сливки', 'сахар', 'десерт', 'торт'],
    ['парить', 'овощи', 'здоровое', 'питание', 'брокколи']
]

In [14]:

# Обучаем модель
model = Word2Vec(
    sentences=cooking_sentences,
    vector_size=50,
    window=3,
    min_count=1,
    workers=2,
    sg=1
)

# Проверяем
print("Похожие на 'варить':")
for word, score in model.wv.most_similar('варить', topn=3):
    print(f"  {word}: {score:.3f}")

Похожие на 'варить':
  вино: 0.240
  ингредиенты: 0.217
  хлеб: 0.194


In [15]:
print(f"Слова в словаре: {list(model.wv.key_to_index.keys())[:10]}...")

Слова в словаре: ['овощи', 'мясо', 'соус', 'вода', 'тесто', 'духовка', 'специи', 'варить', 'брокколи', 'питание']...


4. Проверьте модель

In [16]:
# Проверяем похожие слова в кулинарной тематике
try:
    similar = model.wv.most_similar('варить', topn=5)
    print("Слова, похожие на 'варить':")
    for word, score in similar:
        print(f"  {word}: {score:.4f}")
except KeyError:
    print("Слово 'варить' не найдено в словаре")

Слова, похожие на 'варить':
  вино: 0.2398
  ингредиенты: 0.2172
  хлеб: 0.1938
  брокколи: 0.1846
  кипятить: 0.1711


In [17]:
# Найдите слова, похожие на "духовка"
try:
    similar = model.wv.most_similar('духовка', topn=5)
    print("Слова, похожие на 'духовка':")
    for word, score in similar:
        print(f"  {word}: {score:.4f}")
except KeyError:
    print("Слово 'духовка' не найдено в словаре")

# Найдите слова, похожие на "овощи"
try:
    similar = model.wv.most_similar('овощи', topn=5)
    print("Слова, похожие на 'овощи':")
    for word, score in similar:
        print(f"  {word}: {score:.4f}")
except KeyError:
    print("Слово 'овощи' не найдено в словаре")

Слова, похожие на 'духовка':
  ингредиенты: 0.3199
  десерт: 0.3064
  холодильник: 0.2705
  питание: 0.2243
  пирог: 0.2142
Слова, похожие на 'овощи':
  мариновать: 0.2716
  хлеб: 0.2691
  гриль: 0.2546
  фольга: 0.2409
  сахар: 0.2108


## Часть 2: FastText

FastText улучшает Word2Vec, рассматривая слова как наборы символов (n-грамм). Это позволяет работать с редкими словами и опечатками

5. Обучите FastText на корпусе текстов из пункта 3. Используйте код ниже

In [19]:
ft_model = FastText(
    sentences=cooking_sentences,
    vector_size=50,
    window=3,
    min_count=1,
    workers=2
)

6. Найдите слова, похожие на "варить", "духовка" и "овощи" с помощью обученной модели. Используйте код из пункта 4

In [21]:
print("FastText - похожие на 'варить':")
for word, score in ft_model.wv.most_similar('варить', topn=3):
    print(f"  {word}: {score:.4f}")

print("FastText - похожие на 'духовка':")
for word, score in ft_model.wv.most_similar('духовка', topn=3):
    print(f"  {word}: {score:.4f}")

print("FastText - похожие на 'овощи':")
for word, score in ft_model.wv.most_similar('овощи', topn=3):
    print(f"  {word}: {score:.4f}")

FastText - похожие на 'варить':
  жарить: 0.5353
  парить: 0.4805
  месить: 0.3541
FastText - похожие на 'духовка':
  взбивать: 0.4565
  лимон: 0.3561
  салат: 0.3050
FastText - похожие на 'овощи':
  жарить: 0.2960
  фольга: 0.2574
  морковь: 0.2297


7. Сравните модели

Дана функция для сравнения Word2Vec и FastText

Придумайте 3 слова с опечатками и проверьте, найдет ли их FastText и Word2Vec

In [23]:
def compare_models(word):
    """Сравнивает представления слова в разных моделях"""
    print(f"\nСравнение для слова: '{word}'")

    # Word2Vec
    try:
        w2v_similar = model.wv.most_similar(word, topn=2)
        print(f"  Word2Vec: {[w for w, _ in w2v_similar]}")
    except KeyError:
        print(f"  Word2Vec: слово не найдено")

    # FastText
    try:
        ft_similar = ft_model.wv.most_similar(word, topn=2)
        print(f"  FastText: {[w for w, _ in ft_similar]}")
    except KeyError:
        print(f"  FastText: слово не найдено")

# Сравниваем для разных слов
compare_models('пироги')
compare_models('тесо')
compare_models('мороко')


Сравнение для слова: 'пироги'
  Word2Vec: слово не найдено
  FastText: ['пирог', 'ингредиенты']

Сравнение для слова: 'тесо'
  Word2Vec: слово не найдено
  FastText: ['брокколи', 'яблоки']

Сравнение для слова: 'мороко'
  Word2Vec: слово не найдено
  FastText: ['морковь', 'уголь']


## Часть 3: Doc2Vec

Doc2Vec расширяет Word2Vec для создания векторных представлений целых документов (предложений, абзацев, статей)

In [35]:
# Создаем размеченные документы
documents = [
    "machine learning is interesting",
    "deep learning uses neural networks",
    "python programming for data science",
    "artificial intelligence is amazing",
    "computer vision processes images"
]

# Преобразуем в формат TaggedDocument
tagged_docs = []
for i, doc in enumerate(documents):
    tokens = doc.split()
    tagged_doc = TaggedDocument(words=tokens, tags=[f"doc_{i}"])
    tagged_docs.append(tagged_doc)

print("Размеченные документы:")
for doc in tagged_docs[:5]:
    print(f"  Слова: {doc.words}")
    print(f"  Тег: {doc.tags}")

Размеченные документы:
  Слова: ['machine', 'learning', 'is', 'interesting']
  Тег: ['doc_0']
  Слова: ['deep', 'learning', 'uses', 'neural', 'networks']
  Тег: ['doc_1']
  Слова: ['python', 'programming', 'for', 'data', 'science']
  Тег: ['doc_2']
  Слова: ['artificial', 'intelligence', 'is', 'amazing']
  Тег: ['doc_3']
  Слова: ['computer', 'vision', 'processes', 'images']
  Тег: ['doc_4']


In [25]:
# Обучаем Doc2Vec
doc_model = Doc2Vec(
    documents=tagged_docs,
    vector_size=50,
    window=3,
    min_count=1,
    workers=2,
    epochs=20
)

print("Doc2Vec модель обучена!")
print(f"Количество документов: {len(doc_model.dv.key_to_index)}")

Doc2Vec модель обучена!
Количество документов: 5


In [26]:
# Получаем вектор документа
doc_vector = doc_model.dv["doc_0"]
print(f"Вектор документа doc_0: {doc_vector[:5]}...")

# Находим похожие документы
similar_docs = doc_model.dv.most_similar("doc_0", topn=2)
print("\nДокументы, похожие на doc_0:")
for doc_tag, similarity in similar_docs:
    doc_id = int(doc_tag.split('_')[1])
    print(f"  {doc_tag}: {similarity:.4f}")
    print(f"    Текст: {documents[doc_id]}")

Вектор документа doc_0: [-0.01057    -0.01198188 -0.01982618  0.01710627  0.00710373]...

Документы, похожие на doc_0:
  doc_1: 0.2735
    Текст: deep learning uses neural networks
  doc_2: 0.1275
    Текст: python programming for data science


In [27]:
# Сравниваем схожесть документов
def compare_documents(doc1_id, doc2_id):
    similarity = doc_model.dv.similarity(f"doc_{doc1_id}", f"doc_{doc2_id}")
    print(f"Схожесть doc_{doc1_id} и doc_{doc2_id}: {similarity:.4f}")
    print(f"  doc_{doc1_id}: {documents[doc1_id]}")
    print(f"  doc_{doc2_id}: {documents[doc2_id]}")

compare_documents(0, 1)  # machine learning vs deep learning
compare_documents(0, 3)  # machine learning vs AI

Схожесть doc_0 и doc_1: 0.2735
  doc_0: machine learning is interesting
  doc_1: deep learning uses neural networks
Схожесть doc_0 и doc_3: -0.0822
  doc_0: machine learning is interesting
  doc_3: artificial intelligence is amazing


8. Сравните схожесть doc_2 и doc_4

In [38]:
# Сравниваем схожесть документов
def compare_documents(doc2_id, doc4_id):
    similarity = doc_model.dv.similarity(f"doc_{doc2_id}", f"doc_{doc4_id}")
    print(f"Схожесть doc_{doc2_id} и doc_{doc4_id}: {similarity:.4f}")
    print(f"  doc_{doc2_id}: {documents[doc2_id]}")
    print(f"  doc_{doc4_id}: {documents[doc4_id]}")

compare_documents(2, 4)

Схожесть doc_2 и doc_4: -0.0362
  doc_2: python programming for data science
  doc_4: computer vision processes images


9. Найдите самый похожий документ на doc_1

In [40]:
# Находим самый похожий документ на doc_1
similar_docs = doc_model.dv.most_similar("doc_1")
most_similar_doc = similar_docs[0]

print(f"Самый похожий документ на doc_1:")
print(f"  Документ: {most_similar_doc[0]}")
print(f"  Схожесть: {most_similar_doc[1]:.4f}")
print(f"  Текст: {documents[int(most_similar_doc[0].split('_')[1])]}")

Самый похожий документ на doc_1:
  Документ: doc_0
  Схожесть: 0.2735
  Текст: machine learning is interesting


10. Выберите любую из трёх моделей. Обучите модели с разной размерностью (10, 50, 100). Продемонстрируйте качество их работы на примере поиска похожих слов (выберите любые 3 примера, соответствующих тематике корпуса из пункта 4)

In [41]:
from gensim.models import Word2Vec

# Данные для обучения (кулинарная тематика)
cooking_sentences = [
    ['варить', 'суп', 'овощи', 'морковь', 'картофель'],
    ['жарить', 'курица', 'сковорода', 'масло', 'специи'],
    ['печь', 'хлеб', 'мука', 'дрожжи', 'духовка'],
    ['резать', 'овощи', 'салат', 'помидоры', 'огурцы'],
    ['смешивать', 'ингредиенты', 'тесто', 'яйца', 'молоко'],
    ['варить', 'паста', 'вода', 'соль', 'соус'],
    ['гриль', 'мясо', 'овощи', 'уголь', 'барбекю'],
    ['тушить', 'говядина', 'горшок', 'вино', 'травы'],
    ['запекать', 'рыба', 'лимон', 'духовка', 'фольга'],
    ['готовить', 'завтрак', 'яичница', 'бекон', 'тост']
]

# Обучаем модели с разной размерностью
sizes = [10, 50, 100]
models = {}

for size in sizes:
    print(f"🎯 Обучаем модель с размерностью {size}...")
    models[size] = Word2Vec(
        sentences=cooking_sentences,
        vector_size=size,
        window=3,
        min_count=1,
        workers=2,
        sg=1
    )

# Тестируем на примерах
test_words = ['варить', 'сковорода', 'тесто']

print("\n🔍 СРАВНЕНИЕ КАЧЕСТВА МОДЕЛЕЙ:")
print("=" * 50)

for word in test_words:
    print(f"\n📌 Похожие на '{word}':")
    for size in sizes:
        if word in models[size].wv:
            similar = models[size].wv.most_similar(word, topn=3)
            similar_words = [w for w, _ in similar]
            print(f"  Размер {size:3d}: {similar_words}")
        else:
            print(f"  Размер {size:3d}: слово не найдено")

🎯 Обучаем модель с размерностью 10...
🎯 Обучаем модель с размерностью 50...
🎯 Обучаем модель с размерностью 100...

🔍 СРАВНЕНИЕ КАЧЕСТВА МОДЕЛЕЙ:

📌 Похожие на 'варить':
  Размер  10: ['хлеб', 'дрожжи', 'соус']
  Размер  50: ['хлеб', 'мука', 'морковь']
  Размер 100: ['курица', 'фольга', 'запекать']

📌 Похожие на 'сковорода':
  Размер  10: ['гриль', 'резать', 'бекон']
  Размер  50: ['суп', 'масло', 'говядина']
  Размер 100: ['тесто', 'мука', 'вода']

📌 Похожие на 'тесто':
  Размер  10: ['яйца', 'салат', 'говядина']
  Размер  50: ['печь', 'жарить', 'лимон']
  Размер 100: ['фольга', 'сковорода', 'духовка']
